# Clasificación de textos según los Objetivos de Desarrollo Sostenible

**Micro proyecto 2** · ML No Supervisado (2026-14) · Universidad de los Andes
**Autores:** Juan David Lara Camacho, Miguel
**Entrega:** domingo 20 de septiembre de 2026

Se construye una solución de procesamiento de lenguaje natural que toma un texto en español y lo
relaciona con uno de los Objetivos de Desarrollo Sostenible de la Agenda 2030. El recorrido es
TF-IDF sobre bolsa de palabras, reducción de la dimensionalidad con SVD truncada, que a la vez
sirve de modelo de tópicos por análisis semántico latente, y clasificación sobre esa
representación.

---

### Cómo está organizado este notebook

El orden de las secciones es el de los criterios de evaluación, para que cada bloque se pueda
leer y calificar por separado.

| Sección | Qué construye | Peso |
|---|---|---|
| 1 | Los datos: carga, distribución de clases y partición | |
| 2 | Preparación de los textos y **pipeline** | 30% + 15% |
| 3 | **LSA**: tópicos e interpretación frente a los ODS | 15% |
| 4 | **Clasificación** con búsqueda de hiperparámetros | 30% |
| 5 | Desempeño sobre textos **no vistos** | 10% |
| 6 | Conclusiones | |

## 1. Los datos

El corpus es un subconjunto del **OSDG Community Dataset** en su versión 2023, traducido al
español y aumentado. Tres hechos medidos antes de empezar, que condicionan todo lo que sigue:

1. **Son 16 clases, no 17.** El ODS 17, alianzas para lograr los objetivos, no aparece en el
   archivo. No es un descuido del corpus: en las taxonomías oficiales el ODS 17 es el medio de
   implementación de los demás y no un tema comparable.
2. **Desbalance de 3,5 a 1** entre el ODS 16, con 1.080 textos, y el ODS 12, con 312. La
   exactitud sola es engañosa: la métrica principal es **F1 macro** y la partición va
   estratificada.
3. **Los textos son párrafos** de mediana 105 palabras, entre 24 y 268.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

### 1.1 Carga

In [ ]:
from pathlib import Path

# El corpus se busca junto al notebook o un nivel más arriba, en data/
for ruta in [Path("data/Train_textosODS.xlsx"), Path("../data/Train_textosODS.xlsx")]:
    if ruta.exists():
        break

corpus = pd.read_excel(ruta)
print(f"{corpus.shape[0]} textos, {corpus.shape[1]} columnas: {list(corpus.columns)}")
corpus.head(3)

### 1.2 Distribución de las clases

In [ ]:
conteo = corpus["ODS"].value_counts().sort_index()

fig, eje = plt.subplots(figsize=(10, 4))
eje.bar(conteo.index, conteo.values, color="steelblue")
eje.set_xticks(conteo.index)
eje.set_xlabel("ODS")
eje.set_ylabel("textos")
eje.set_title("Distribución de los textos por Objetivo de Desarrollo Sostenible")
eje.grid(axis="y", alpha=0.3)
plt.show()

print(f"clases presentes: {corpus['ODS'].nunique()} de 17   "
      f"(falta el ODS {sorted(set(range(1, 18)) - set(corpus['ODS'].unique()))})")
print(f"desbalance mayor/menor: {conteo.max() / conteo.min():.2f} a 1")

### 1.3 Partición estratificada

Se reserva un conjunto de prueba que **no se toca** hasta la sección 5. La estratificación
mantiene la proporción de cada ODS en las dos partes, que con este desbalance no es opcional.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    corpus["textos"].astype(str),
    corpus["ODS"],
    test_size=0.2,
    stratify=corpus["ODS"],
    random_state=RANDOM_STATE,
)

print(f"entrenamiento: {len(X_train)} textos")
print(f"prueba:        {len(X_test)} textos")

---

## 2. Preparación de los textos y pipeline

> **Criterios:** *preparación de los datos, incluida la reducción de la dimensionalidad,
> justificando las decisiones* (**30%**) y *construcción del pipeline de preparación*
> (**15%**).

Lo que hay que resolver aquí, con su argumento escrito al lado de cada decisión:

- **Normalización.** Palabras vacías del español, que `TfidfVectorizer` no trae; qué se hace con
  tildes, mayúsculas, números y signos; si se lematiza o se aplica *stemming*.
- **Vectorización.** Rango de n-gramas, `min_df`, `max_df`, `sublinear_tf`, `max_features`.
- **Pipeline.** Todo lo anterior dentro de un `Pipeline` de scikit-learn, para que el mismo
  objeto procese los textos nuevos. Este es el criterio del 15% y se pierde entero si se
  vectoriza a mano en celdas sueltas.

Las decisiones 1 y 2 de [`docs/decisiones.md`](../docs/decisiones.md) corresponden a esta
sección.

### 2.1 Normalización del texto

### 2.2 Vectorización TF-IDF

### 2.3 El pipeline

### 2.4 Línea base, sin reducción de la dimensionalidad

Antes de proyectar conviene medir cuánto rinde el TF-IDF crudo con un clasificador lineal. Ese
número es la vara contra la cual se compara todo lo demás: si después de la SVD el desempeño cae
mucho, hay que decirlo y explicar por qué se aplica igual.

En la investigación previa esta línea base dio **0,8886 de exactitud y 0,8659 de F1 macro** con
validación cruzada de cinco particiones sobre el corpus completo. Ver
[`docs/estrategia.md`](../docs/estrategia.md).

---

## 3. Modelo de tópicos con LSA

> **Criterio:** *construcción del modelo LSA sobre la matriz TF-IDF e interpretación cualitativa
> de al menos 5 tópicos frente a los ODS* (**15%**).

El enunciado pide `TruncatedSVD` sobre la matriz TF-IDF, explorando un número reducido de
componentes, entre 10 y 20, y mostrar para **al menos cinco** de ellas las palabras de mayor
peso. La interpretación cualitativa es lo que se califica, no la tabla de *loadings*.

**Dos cosas que conviene tener presentes.** La primera es de orientación: `svd.components_` tiene
forma (componentes × términos) y es la que da las palabras de cada tópico, mientras que
`svd.transform(X)` tiene forma (documentos × componentes) y da la representación de cada
documento. La segunda es que **probablemente los tópicos no se alineen uno a uno con los ODS**, y
eso no es un fracaso: la estructura que hay en el lenguaje tiene tres o cinco bloques temáticos,
no dieciséis. El argumento está en [`docs/estrategia.md`](../docs/estrategia.md), sección 3.

La decisión 3 de [`docs/decisiones.md`](../docs/decisiones.md) corresponde a esta sección.

### 3.1 SVD truncada

### 3.2 Cuántas componentes: varianza explicada y desempeño

### 3.3 Los tópicos, término por término

### 3.4 Interpretación frente a los ODS

*(Aquí va la lectura de al menos cinco componentes: qué tema reconocible forma cada una y con qué
ODS se corresponde, o con qué bloque de ODS si no hay correspondencia uno a uno.)*

---

## 4. Modelo de clasificación

> **Criterio:** *modelo de clasificación con el algoritmo seleccionado con búsqueda de
> hiperparámetros, validándolo con medidas de evaluación adecuadas. Se justifica la selección del
> algoritmo y las métricas empleadas. Se aplica un método de reducción de la dimensionalidad*
> (**30%**).

Tres exigencias literales, y las tres se califican:

1. **Justificar el algoritmo**, no solo elegirlo.
2. **Búsqueda de hiperparámetros explícita**, con `GridSearchCV` o `RandomizedSearchCV`.
3. **Métricas adecuadas y justificadas.** Con 16 clases desbalanceadas, la principal es F1 macro
   y la validación cruzada va estratificada.

La decisión 4 de [`docs/decisiones.md`](../docs/decisiones.md) corresponde a esta sección.

### 4.1 Elección del algoritmo

### 4.2 Búsqueda de hiperparámetros

### 4.3 Evaluación: métricas y matriz de confusión

---

## 5. Desempeño sobre textos no vistos

> **Criterio:** *evidencia del desempeño del método construido mostrando las clasificaciones
> sobre un conjunto de textos que no hayan sido utilizados durante el aprendizaje* (**10%**).

Hay que mostrar **al menos cuatro textos** del conjunto de prueba, uno por uno, con su ODS real y
el predicho. Es un criterio que se pierde entero por olvido, así que va antes de las conclusiones
y no al final.

Vale la pena mostrar también la confianza del modelo y su segunda opción: en la investigación
previa, el 62% de los errores tenían el ODS correcto en segunda posición, y el top-2 sube de
0,8886 a 0,9572.

---

## 6. Conclusiones

*(Qué se construyó, qué desempeño alcanza, qué decisiones lo explican y cuáles son sus límites.)*

---

### Antes de exportar

- [ ] El notebook corre de punta a punta, en orden, sin errores
- [ ] Cada decisión tiene su justificación escrita al lado, no solo el código
- [ ] El pipeline es un objeto de scikit-learn y procesa un texto nuevo de principio a fin
- [ ] Hay búsqueda de hiperparámetros, y se dice por qué ese espacio y esa métrica
- [ ] Se interpretan al menos cinco componentes frente a los ODS
- [ ] Se muestran al menos cuatro textos de prueba clasificados
- [ ] Se dice que son 16 clases y no 17, y por qué
- [ ] Se declara que el corpus está traducido automáticamente y aumentado
- [ ] Todas las celdas quedan con su salida visible